In [1]:
# =====================================================================
# LIBRARY IMPORT
# =====================================================================

import os
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm  # For progress bars
import gc  # For garbage collection
import re  # For text cleaning
from collections import defaultdict
from datetime import datetime

In [2]:
# =====================================================================
# CONFIGURATION SECTION - Toggle between sample and full processing
# =====================================================================

# Set to True for sample mode, False for full processing
# SAMPLE_MODE = True  # <-- TOGGLE HERE between sample and full processing
SAMPLE_MODE = False  # <-- TOGGLE HERE between sample and full processing

# Sample configuration
MAX_FILES_TO_PROCESS = 2      # Number of files to process in sample mode
MAX_ITEMS_PER_FILE = 1000     # Maximum items to extract from each file

# Batch size for memory management
BATCH_SIZE = 5000  # Larger batch size for full processing
SAMPLE_BATCH_SIZE = 1000  # Smaller batch size for sample mode

# Use appropriate batch size based on mode
batch_size = SAMPLE_BATCH_SIZE if SAMPLE_MODE else BATCH_SIZE

print(f"Running in {'SAMPLE' if SAMPLE_MODE else 'FULL'} mode")
print(f"Batch size: {batch_size}")
if SAMPLE_MODE:
    print(f"Will process max {MAX_FILES_TO_PROCESS} files, {MAX_ITEMS_PER_FILE} items per file")


Running in FULL mode
Batch size: 5000


In [3]:
# =====================================================================
# DATA LOADING
# =====================================================================

# Your specific dataset path
input_dir = '/kaggle/input/product-listings'
print(f"Using input directory: {input_dir}")

# Get all the JSON files from your directory
json_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.json')]

# In sample mode, limit the number of files
if SAMPLE_MODE and len(json_files) > MAX_FILES_TO_PROCESS:
    json_files = json_files[:MAX_FILES_TO_PROCESS]
    
print(f"Found {len(json_files)} JSON files to process: {[os.path.basename(f) for f in json_files]}")


Using input directory: /kaggle/input/product-listings
Found 16 JSON files to process: ['listings_3.json', 'listings_d.json', 'listings_b.json', 'listings_a.json', 'listings_4.json', 'listings_8.json', 'listings_9.json', 'listings_f.json', 'listings_0.json', 'listings_6.json', 'listings_c.json', 'listings_2.json', 'listings_5.json', 'listings_1.json', 'listings_e.json', 'listings_7.json']


In [4]:
def extract_english_value(field_list, join=True):
    """Extract English values from a language-tagged field list"""
    if not isinstance(field_list, list):
        return np.nan
    
    values = []
    for item in field_list:
        if isinstance(item, dict) and 'language_tag' in item and item['language_tag'].startswith('en_'):
            if 'value' in item and item['value']:
                values.append(str(item['value']))
    
    if not values:
        return np.nan
        
    return ' | '.join(values) if join else values

def has_english_content(data):
    """Check if the item has any English content in item_name field"""
    if 'item_name' in data and isinstance(data['item_name'], list):
        for item in data['item_name']:
            if isinstance(item, dict) and 'language_tag' in item and item['language_tag'].startswith('en_'):
                return True
    return False

def extract_color_standardized(color_data):
    """Extract standardized color values for English entries"""
    if not isinstance(color_data, list):
        return np.nan
    
    for color_item in color_data:
        if isinstance(color_item, dict) and color_item.get('language_tag', '').startswith('en_'):
            std_values = color_item.get('standardized_values', [])
            if std_values:
                return std_values[0]  # Take first standardized value
    
    return np.nan

def extract_node_name(node_list):
    """Extract node name from the first node"""
    if isinstance(node_list, list) and len(node_list) > 0:
        if isinstance(node_list[0], dict) and 'node_name' in node_list[0]:
            return node_list[0]['node_name']
    return np.nan

# Main processing function
def process_json_files(files, batch_size=1000, sample_mode=False, max_items_per_file=None):
    """Process JSON files to extract specified metadata fields"""
    all_items = []
    total_processed = 0
    
    for file_path in tqdm(files, desc="Processing files"):
        batch = []
        items_from_file = 0
        try:
            with open(file_path, 'r') as f:
                for line_num, line in enumerate(f, 1):
                    try:
                        # In sample mode, limit items per file
                        if sample_mode and max_items_per_file and items_from_file >= max_items_per_file:
                            print(f"Reached sample limit of {max_items_per_file} items for file {os.path.basename(file_path)}")
                            break
                        
                        obj = json.loads(line.strip())
                        # Only keep items with English content in item_name
                        if has_english_content(obj):
                            batch.append(obj)
                            items_from_file += 1
                        
                        # Process in batches to save memory
                        if len(batch) >= batch_size:
                            all_items.extend(batch)
                            total_processed += len(batch)
                            print(f"Processed {total_processed} items so far...")
                            batch = []
                            gc.collect()  # Force garbage collection
                            
                    except json.JSONDecodeError:
                        print(f"JSON decode error in file {os.path.basename(file_path)}, line {line_num}")
                    except Exception as e:
                        print(f"Error processing line {line_num} in {os.path.basename(file_path)}: {e}")
                
                # Add any remaining items in the last batch
                if batch:
                    all_items.extend(batch)
                    total_processed += len(batch)
                    batch = []
                    gc.collect()
                    
        except Exception as e:
            print(f"Error opening file {os.path.basename(file_path)}: {e}")
            
    print(f"Total items loaded: {len(all_items)}")
    return all_items

# Process the data into a structured format with English-only fields
def extract_metadata(all_items):
    """Extract specified metadata fields from the items"""
    print("Extracting metadata fields...")
    flattened_data = []
    
    for item in tqdm(all_items, desc="Extracting metadata"):
        # Extract basic fields
        flat_item = {
            'item_id': item.get('item_id', np.nan),
            'main_image_id': item.get('main_image_id', np.nan),
            'item_name': extract_english_value(item.get('item_name', []), join=False),
            'brand': extract_english_value(item.get('brand', []), join=False),
            'bullet_point': extract_english_value(item.get('bullet_point', [])),
            'item_keywords': extract_english_value(item.get('item_keywords', [])),
            'material': extract_english_value(item.get('material', []), join=False),
            'node_name': extract_node_name(item.get('node', [])),
            'color_standardized': extract_color_standardized(item.get('color', [])),
        }
        
        # Extract other specified fields (these don't need language filtering)
        if 'product_type' in item and isinstance(item['product_type'], list) and len(item['product_type']) > 0:
            if isinstance(item['product_type'][0], dict) and 'value' in item['product_type'][0]:
                flat_item['product_type'] = item['product_type'][0]['value']
        
        # Extract the remaining metadata fields with English values
        for field_name in ['model_name', 'style', 'fabric_type', 'pattern', 
                          'item_shape', 'finish_type', 'product_description']:
            if field_name in item:
                flat_item[field_name] = extract_english_value(item.get(field_name, []), join=False)
        
        # Convert any list values to joined strings for consistency
        for key, value in flat_item.items():
            if isinstance(value, list):
                flat_item[key] = ' | '.join(value) if value else np.nan
        
        flattened_data.append(flat_item)
    
    # Create DataFrame
    return pd.DataFrame(flattened_data)

In [5]:
# =====================================================================
# MAIN PROCESSING
# =====================================================================

# Process JSON files with appropriate parameters
print("Loading and processing product listings...")
all_items = process_json_files(
    json_files, 
    batch_size=batch_size,
    sample_mode=SAMPLE_MODE,
    max_items_per_file=MAX_ITEMS_PER_FILE if SAMPLE_MODE else None
)

# Extract metadata fields from the processed items
print("Extracting metadata and creating DataFrame...")
df = extract_metadata(all_items)

# Free up memory from intermediate data structure
del all_items
gc.collect()

# =====================================================================
# DATA QUALITY CHECKS AND ANALYSIS
# =====================================================================

# Check data shape and quality
print("\nChecking data quality...")
print(f"DataFrame shape: {df.shape}")

# Define text columns for analysis
text_columns = ['item_name', 'brand', 'bullet_point', 'item_keywords',
               'material', 'model_name', 'style', 'fabric_type',
               'pattern', 'item_shape', 'finish_type', 'product_description']
present_text_columns = [col for col in text_columns if col in df.columns]

# Calculate true column completeness BEFORE filling NA values
true_completeness = (df.count() / len(df) * 100).sort_values(ascending=False)
print("\nTrue column completeness (% non-null):")
for col, pct in true_completeness.items():
    print(f"{col}: {pct:.1f}%")

# Calculate empty string counts for text columns
print("\nEmpty string analysis for text columns:")
for col in present_text_columns:
    if col in df.columns:
        # Count empty strings (after removing leading/trailing whitespace)
        empty_count = (df[col].fillna('').str.strip() == '').sum()
        empty_percent = empty_count / len(df) * 100
        print(f"{col}: {empty_count} empty strings ({empty_percent:.1f}%)")

# =====================================================================
# DATA PREPARATION FOR PROCESSING
# =====================================================================

# Create a working copy to preserve the original data with nulls
df_processed = df.copy()

# Fill missing values for text columns with empty strings for processing
for col in present_text_columns:
    df_processed[col] = df_processed[col].fillna('')

# =====================================================================
# SAVE DATA
# =====================================================================

# Create output file names with appropriate prefixes
sample_prefix = "sample_" if SAMPLE_MODE else ""
output_dir = '/kaggle/working'

# Save as CSV with English-only content (original with nulls preserved)
csv_filename = f'{output_dir}/{sample_prefix}abo_products_english_metadata.csv'
df.to_csv(csv_filename, index=False)
print(f"\nSaved metadata CSV to: {csv_filename}")

# Also save as a pickle file for faster future loading (preserves data types)
pickle_filename = f'{output_dir}/{sample_prefix}abo_products_english_metadata.pkl'
df.to_pickle(pickle_filename)
print(f"Saved metadata pickle file to: {pickle_filename}")

# Generate a summary file with dataset statistics
summary_filename = f'{output_dir}/{sample_prefix}metadata_summary.txt'
with open(summary_filename, 'w') as f:
    f.write("# ABO PRODUCT METADATA SUMMARY\n")
    f.write(f"Generated on: {pd.Timestamp.now()}\n\n")
    
    f.write(f"Total products: {len(df)}\n")
    f.write(f"Dataset shape: {df.shape}\n\n")
    
    f.write("## Column Completeness\n")
    for col, pct in true_completeness.items():
        f.write(f"{col}: {pct:.1f}%\n")
    
    f.write("\n## Product Types\n")
    if 'product_type' in df.columns:
        type_counts = df['product_type'].value_counts().head(20)
        for product_type, count in type_counts.items():
            f.write(f"{product_type}: {count} ({count/len(df)*100:.1f}%)\n")
    
    f.write("\n## Field Statistics\n")
    for col in present_text_columns:
        # Calculate length statistics for actually non-empty text fields
        # Use the original df to get accurate counts
        non_empty_mask = df[col].notna() & (df[col].str.strip() != '')
        non_empty_count = non_empty_mask.sum()
        
        if non_empty_count > 0:
            # Get lengths of non-empty values
            lengths = df.loc[non_empty_mask, col].str.len()
            
            f.write(f"\n{col}:\n")
            f.write(f"  Non-empty count: {non_empty_count}\n")
            f.write(f"  Mean length: {lengths.mean():.1f} chars\n")
            f.write(f"  Median length: {lengths.median():.1f} chars\n")
            f.write(f"  Max length: {lengths.max()} chars\n")

print(f"Saved metadata summary to: {summary_filename}")

# Print a small sample of the data
print("\nSample of extracted metadata (first 3 rows):")
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 150)
print(df.head(3))

print("\nProcessing completed successfully!")

Loading and processing product listings...


Processing files:   0%|          | 0/16 [00:00<?, ?it/s]

Processed 5000 items so far...
Processed 12690 items so far...
Processed 20357 items so far...
Processed 28058 items so far...
Processed 35706 items so far...
Processed 43440 items so far...
Processed 51170 items so far...
Processed 58829 items so far...
Processed 66491 items so far...
Processed 74175 items so far...
Processed 81851 items so far...
Processed 89527 items so far...
Processed 97204 items so far...
Processed 104873 items so far...
Processed 112525 items so far...
Processed 120131 items so far...
Total items loaded: 122734
Extracting metadata and creating DataFrame...
Extracting metadata fields...


Extracting metadata:   0%|          | 0/122734 [00:00<?, ?it/s]


Checking data quality...
DataFrame shape: (122734, 17)

True column completeness (% non-null):
item_id: 100.0%
item_name: 100.0%
product_type: 100.0%
main_image_id: 99.6%
brand: 96.6%
node_name: 95.0%
bullet_point: 92.5%
item_keywords: 87.6%
color_standardized: 70.6%
model_name: 60.8%
material: 38.2%
style: 24.8%
fabric_type: 4.5%
item_shape: 3.2%
pattern: 2.8%
product_description: 1.6%
finish_type: 1.1%

Empty string analysis for text columns:
item_name: 0 empty strings (0.0%)
brand: 4148 empty strings (3.4%)
bullet_point: 9258 empty strings (7.5%)
item_keywords: 15170 empty strings (12.4%)
material: 75795 empty strings (61.8%)
model_name: 48060 empty strings (39.2%)
style: 92329 empty strings (75.2%)
fabric_type: 117181 empty strings (95.5%)
pattern: 119259 empty strings (97.2%)
item_shape: 118791 empty strings (96.8%)
finish_type: 121379 empty strings (98.9%)
product_description: 120729 empty strings (98.4%)

Saved metadata CSV to: /kaggle/working/abo_products_english_metadata.csv


In [6]:
# =====================================================================
# DATA ANALYSIS AND OUTPUTS
# =====================================================================


# Print summary statistics
print(f"\nDataFrame shape: {df.shape}")
print(f"Number of columns: {len(df.columns)}")
print("\nColumn names:")
print(df.columns.tolist())


DataFrame shape: (122734, 17)
Number of columns: 17

Column names:
['item_id', 'main_image_id', 'item_name', 'brand', 'bullet_point', 'item_keywords', 'material', 'node_name', 'color_standardized', 'product_type', 'model_name', 'style', 'fabric_type', 'pattern', 'item_shape', 'finish_type', 'product_description']


In [8]:

# === 📊 Utility: Show product type breakdown in table form ===
def print_category_table(df, label, top_n=20):
    counts = df['product_type'].value_counts().head(top_n)
    percentages = (counts / len(df) * 100).round(1)
    table_df = pd.DataFrame({
        'Product Type': counts.index,
        'Count': counts.values,
        'Percentage': [f"{p}%" for p in percentages]
    })
    print(f"\n## {label}")
    print(table_df.to_string(index=False))

# === 📋 Utility: Field stats with word-level metrics ===
def compute_field_stats(df, title="# Field Statistics (Word-Level)"):
    print(f"\n{title}\n")
    summary_data = []
    total_rows = len(df)

    for col in df.columns:
        dtype = df[col].dtype
        non_empty_count = df[col].notna().sum()
        non_empty_pct = round((non_empty_count / total_rows) * 100, 1)

        if dtype == 'object' or dtype == 'string':
            word_counts = df[col].dropna().astype(str).map(lambda x: len(x.split()))
            mean_words = round(word_counts.mean(), 1) if not word_counts.empty else ""
            median_words = round(word_counts.median(), 1) if not word_counts.empty else ""
            max_words = word_counts.max() if not word_counts.empty else ""
        else:
            mean_words = median_words = max_words = ""

        summary_data.append([
            col, dtype,
            f"{non_empty_count:,}", f"{non_empty_pct}%",
            mean_words, median_words, max_words
        ])

    summary_df = pd.DataFrame(summary_data, columns=[
        "Field", "Data Type", "Non-empty Count", "Non-empty (%)",
        "Mean Word Count", "Median Word Count", "Max Word Count"
    ])
    print(summary_df.to_string(index=False))

In [9]:

# === 🧾 BEFORE CAPPING: Show stats on full dataset ===
print_category_table(df, "Top 20 Product Types (Before Capping)")
compute_field_stats(df, title="# Field Statistics (Before Capping)")



## Top 20 Product Types (Before Capping)
              Product Type  Count Percentage
       CELLULAR_PHONE_CASE  64757      52.8%
                     SHOES   9970       8.1%
                   GROCERY   6214       5.1%
                      HOME   2642       2.2%
                     CHAIR   1714       1.4%
         HOME_BED_AND_BATH   1664       1.4%
  HOME_FURNITURE_AND_DECOR   1435       1.2%
      HEALTH_PERSONAL_CARE   1207       1.0%
                      BOOT   1161       0.9%
                    SANDAL   1015       0.8%
                      SOFA    998       0.8%
FINENECKLACEBRACELETANKLET    965       0.8%
                  FINERING    962       0.8%
                 ACCESSORY    863       0.7%
                       RUG    817       0.7%
                     TABLE    747       0.6%
               FINEEARRING    744       0.6%
                   HANDBAG    700       0.6%
           OFFICE_PRODUCTS    679       0.6%
                  WALL_ART    599       0.5%

# Field Stat

In [10]:
# === ✅ STEP 1: Filter rows where main_image_id is present
df_capped = df[df['main_image_id'].notna()].reset_index(drop=True)

# === ✅ STEP 2: Cap product_type to 10% max
MAX_RATIO = 0.10
initial_max = int(len(df_capped) * MAX_RATIO)
estimated_total = sum(min(count, initial_max) for count in df_capped['product_type'].value_counts())
final_max = int(estimated_total * MAX_RATIO)

capped_df_list, drop_log = [], defaultdict(int)

for category, group in df_capped.groupby('product_type'):
    if len(group) > final_max:
        drop_log[category] = len(group) - final_max
        group = group.sample(n=final_max, random_state=42)
    capped_df_list.append(group)

df_capped = pd.concat(capped_df_list).reset_index(drop=True)
df_capped = df_capped.sample(frac=1, random_state=42).reset_index(drop=True)

# === ✅ STEP 3: Truncate item_keywords to 256 words
df_capped['item_keywords'] = df_capped['item_keywords'].apply(
    lambda x: ' '.join(x.split()[:256]) if isinstance(x, str) else x
)

# === ✅ STEP 4: Rename column
df_capped = df_capped.rename(columns={'main_image_id': 'image_id'})

# === 🧾 AFTER CAPPING: Show capped category and updated field stats
print_category_table(df_capped, "Top 20 Product Types (After Capping)")
compute_field_stats(df_capped, title="# Field Statistics (After Capping & Truncation)")

# === ✅ STEP 5: Log dropped rows
if drop_log:
    print("\n## Dropped Items Per Category (Before vs After):")
    for category, dropped in drop_log.items():
        print(f"{category}: {dropped:,} dropped")


## Top 20 Product Types (After Capping)
              Product Type  Count Percentage
       CELLULAR_PHONE_CASE   6977      11.3%
                     SHOES   6977      11.3%
                   GROCERY   6177      10.0%
                      HOME   2638       4.3%
                     CHAIR   1713       2.8%
         HOME_BED_AND_BATH   1662       2.7%
  HOME_FURNITURE_AND_DECOR   1376       2.2%
      HEALTH_PERSONAL_CARE   1168       1.9%
                      BOOT   1161       1.9%
                    SANDAL   1015       1.6%
                      SOFA    997       1.6%
FINENECKLACEBRACELETANKLET    965       1.6%
                  FINERING    962       1.6%
                 ACCESSORY    828       1.3%
                       RUG    817       1.3%
                     TABLE    747       1.2%
               FINEEARRING    744       1.2%
                   HANDBAG    682       1.1%
           OFFICE_PRODUCTS    679       1.1%
                  WALL_ART    598       1.0%

# Field Stati

In [11]:
# === ✅ STEP 6: Save final capped dataset
capped_output_csv = f"{output_dir}/{sample_prefix}abo_products_capped_10percent.csv"
df_capped.to_csv(capped_output_csv, index=False)
print(f"\nSaved capped dataset to: {capped_output_csv}")


Saved capped dataset to: /kaggle/working/abo_products_capped_10percent.csv


DO NOT GO THERE

In [ ]:
# # =====================================================================
# # DATA ANALYSIS AND OUTPUTS
# # =====================================================================


# # Print summary statistics
# print(f"\nDataFrame shape: {df.shape}")
# print(f"Number of columns: {len(df.columns)}")
# print("\nColumn names:")
# print(df.columns.tolist())

In [ ]:
# # Calculate column non-null percentages
# non_null_percent = (df.count() / len(df) * 100).sort_values(ascending=False)
# print("\nColumns with highest non-null percentages:")
# print(non_null_percent.head(10))

# print("\nColumns with lowest non-null percentages:")
# print(non_null_percent.tail(10))

In [ ]:
# # Display sample of the data
# print("\nSample data (text fields):")
# pd.set_option('display.max_columns', 5)
# pd.set_option('display.max_colwidth', 50)
# text_columns = ['item_id', 'item_name', 'brand', 'bullet_point', 'product_description']
# text_columns = [col for col in text_columns if col in df.columns]
# print(df[text_columns].sample(min(3, len(df))))

In [ ]:
# # =====================================================================
# # QUANTITATIVE ANALYSIS
# # =====================================================================
# # Add this code after creating your DataFrame (df) but before saving the files

# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from collections import Counter
# import pandas as pd

# print("\n" + "="*80)
# print("QUANTITATIVE ANALYSIS OF PRODUCT TYPES AND TEXT FIELDS")
# print("="*80)

# # -------------------------------------------------
# # 1. Product Type Distribution Analysis
# # -------------------------------------------------
# print("\n1. PRODUCT TYPE DISTRIBUTION")
# print("-" * 40)

# # Count frequency of each product type
# product_type_counts = df['product_type'].value_counts()
# total_products = len(df)

# # Display top product types and their percentages
# print(f"Total unique product types: {len(product_type_counts)}")
# print("\nTop 20 product types:")
# for idx, (product_type, count) in enumerate(product_type_counts.head(20).items()):
#     percentage = (count / total_products) * 100
#     print(f"{idx+1}. {product_type}: {count} products ({percentage:.2f}%)")

# # Calculate diversity metrics
# top_3_percentage = (product_type_counts.head(3).sum() / total_products) * 100
# top_10_percentage = (product_type_counts.head(10).sum() / total_products) * 100
# print(f"\nTop 3 product types account for {top_3_percentage:.2f}% of all products")
# print(f"Top 10 product types account for {top_10_percentage:.2f}% of all products")

# # Calculate Gini coefficient to measure inequality in distribution
# def gini_coefficient(x):
#     # Mean absolute difference
#     mad = np.abs(np.subtract.outer(x, x)).mean()
#     # Relative mean absolute difference
#     rmad = mad / np.mean(x)
#     # Gini coefficient
#     return 0.5 * rmad

# gini = gini_coefficient(product_type_counts.values)
# print(f"Gini coefficient for product type distribution: {gini:.4f}")
# print("(0 = perfect equality, 1 = perfect inequality)")

# # Create a pie chart for the top categories
# plt.figure(figsize=(12, 8))
# top_n = 7  # Show top 7 categories, group the rest as "Other"

# # Prepare data for pie chart
# if len(product_type_counts) > top_n:
#     top_counts = product_type_counts.head(top_n)
#     other_count = product_type_counts.iloc[top_n:].sum()
    
#     labels = list(top_counts.index) + ['Other']
#     sizes = list(top_counts.values) + [other_count]
# else:
#     labels = product_type_counts.index
#     sizes = product_type_counts.values

# # Define custom colors
# colors = sns.color_palette("viridis", len(labels))

# # Create pie chart
# plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
# plt.axis('equal')
# plt.title('Product Type Distribution', fontsize=16)
# plt.tight_layout()

# # Save the pie chart
# pie_chart_path = '/kaggle/working/product_type_distribution.png'
# plt.savefig(pie_chart_path)
# print(f"\nPie chart saved to: {pie_chart_path}")
# plt.close()

# # Create a horizontal bar chart for top 15 categories
# plt.figure(figsize=(12, 10))
# top_15 = product_type_counts.head(15)
# bars = plt.barh(top_15.index, top_15.values, color=sns.color_palette("viridis", len(top_15)))

# # Add percentage labels
# for i, bar in enumerate(bars):
#     percentage = (top_15.values[i] / total_products) * 100
#     plt.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
#              f'{percentage:.1f}%', va='center')

# plt.title('Top 15 Product Types', fontsize=16)
# plt.xlabel('Number of Products', fontsize=12)
# plt.ylabel('Product Type', fontsize=12)
# plt.tight_layout()

# # Save the bar chart
# bar_chart_path = '/kaggle/working/top_product_types.png'
# plt.savefig(bar_chart_path)
# print(f"Bar chart saved to: {bar_chart_path}")
# plt.close()


In [ ]:

# # -------------------------------------------------
# # 2. Text Field Length Analysis
# # -------------------------------------------------
# print("\n2. TEXT FIELD LENGTH CHARACTERISTICS")
# print("-" * 40)

# # Define text fields to analyze
# text_fields = ['item_name', 'brand', 'bullet_point', 'product_description', 'item_keywords']

# # Calculate length for each field
# for field in text_fields:
#     if field in df.columns:
#         # Create a new column with text lengths
#         length_col = f'{field}_length'
#         df[length_col] = df[field].fillna('').astype(str).str.len()
        
#         # Get basic statistics
#         mean_len = df[length_col].mean()
#         median_len = df[length_col].median()
#         min_len = df[length_col].min()
#         max_len = df[length_col].max()
#         std_len = df[length_col].std()
        
#         # Calculate percentiles
#         p25 = df[length_col].quantile(0.25)
#         p75 = df[length_col].quantile(0.75)
#         p90 = df[length_col].quantile(0.90)
#         p95 = df[length_col].quantile(0.95)
#         p99 = df[length_col].quantile(0.99)
        
#         # Count non-empty values
#         non_empty = (df[length_col] > 0).sum()
#         percentage = (non_empty / len(df)) * 100
        
#         print(f"\n{field} Length Statistics:")
#         print(f"  Availability: {non_empty}/{len(df)} ({percentage:.2f}%)")
#         print(f"  Mean: {mean_len:.2f} chars")
#         print(f"  Median: {median_len:.2f} chars")
#         print(f"  Standard Deviation: {std_len:.2f} chars")
#         print(f"  Min: {min_len} chars")
#         print(f"  Max: {max_len} chars")
#         print(f"  25th percentile: {p25:.2f} chars")
#         print(f"  75th percentile: {p75:.2f} chars")
#         print(f"  90th percentile: {p90:.2f} chars")
#         print(f"  95th percentile: {p95:.2f} chars")
#         print(f"  99th percentile: {p99:.2f} chars")
#     else:
#         print(f"\n{field} not found in dataset")

# # Create histograms of text field lengths
# plt.figure(figsize=(15, 10))

# for i, field in enumerate(text_fields):
#     if field in df.columns:
#         length_col = f'{field}_length'
        
#         # Skip if all values are missing
#         if df[length_col].max() == 0:
#             continue
            
#         plt.subplot(len(text_fields), 1, i+1)
        
#         # Use log scale for x-axis if the range is very large
#         max_len = df[length_col].max()
#         if max_len > 1000:
#             # For very skewed distributions, plot with log scale
#             bins = np.logspace(0, np.log10(max_len+1), 50)
#             plt.xscale('log')
#             plt.hist(df[length_col][df[length_col] > 0], bins=bins, alpha=0.7)
#         else:
#             # Regular histogram for less skewed distributions
#             plt.hist(df[length_col], bins=30, alpha=0.7)
            
#         plt.title(f'{field} Length Distribution')
#         plt.xlabel('Character Length (log scale if range > 1000)')
#         plt.ylabel('Frequency')
#         plt.grid(True, alpha=0.3)

# plt.tight_layout()
# text_lengths_chart_path = '/kaggle/working/text_field_lengths.png'
# plt.savefig(text_lengths_chart_path)
# print(f"\nText field length histograms saved to: {text_lengths_chart_path}")
# plt.close()

# # Create a boxplot to compare all text fields
# plt.figure(figsize=(14, 8))

# # Prepare data for boxplot
# boxplot_data = []
# boxplot_labels = []

# for field in text_fields:
#     if field in df.columns:
#         length_col = f'{field}_length'
#         # Only include non-zero values
#         field_data = df[length_col][df[length_col] > 0]
        
#         if not field_data.empty:
#             boxplot_data.append(field_data)
#             boxplot_labels.append(field)

# # Create the boxplot with a log scale
# plt.boxplot(boxplot_data, labels=boxplot_labels, showfliers=False)
# plt.yscale('log')
# plt.title('Text Field Length Comparison (log scale)', fontsize=16)
# plt.xlabel('Field Name', fontsize=12)
# plt.ylabel('Character Length (log scale)', fontsize=12)
# plt.grid(True, alpha=0.3, axis='y')
# plt.xticks(rotation=45)

# plt.tight_layout()
# boxplot_path = '/kaggle/working/text_field_boxplot.png'
# plt.savefig(boxplot_path)
# print(f"Text field boxplot saved to: {boxplot_path}")
# plt.close()

# # -------------------------------------------------
# # 3. Word Count Analysis for Text Fields
# # -------------------------------------------------
# print("\n3. WORD COUNT ANALYSIS")
# print("-" * 40)

# # Calculate word counts for each text field
# for field in text_fields:
#     if field in df.columns:
#         # Create a new column with word counts
#         word_count_col = f'{field}_word_count'
#         df[word_count_col] = df[field].fillna('').astype(str).str.split().str.len()
        
#         # Get basic statistics
#         mean_words = df[word_count_col].mean()
#         median_words = df[word_count_col].median()
#         min_words = df[word_count_col].min()
#         max_words = df[word_count_col].max()
        
#         # Calculate percentiles
#         p90_words = df[word_count_col].quantile(0.90)
#         p95_words = df[word_count_col].quantile(0.95)
#         p99_words = df[word_count_col].quantile(0.99)
        
#         print(f"\n{field} Word Count Statistics:")
#         print(f"  Mean: {mean_words:.2f} words")
#         print(f"  Median: {median_words:.2f} words")
#         print(f"  Min: {min_words} words")
#         print(f"  Max: {max_words} words")
#         print(f"  90th percentile: {p90_words:.2f} words")
#         print(f"  95th percentile: {p95_words:.2f} words")
#         print(f"  99th percentile: {p99_words:.2f} words")
        
#         # Recommended max token limit based on 95th percentile
#         # Assuming avg of 1.3 tokens per word for English
#         token_limit = int(p95_words * 1.3)
#         print(f"  Recommended token limit: {token_limit} tokens (based on 95th percentile)")
#     else:
#         print(f"\n{field} not found in dataset")


In [ ]:
# # -------------------------------------------------
# # 4. Generate Recommendations Based on Analysis
# # -------------------------------------------------
# print("\n4. DATA-DRIVEN RECOMMENDATIONS")
# print("-" * 40)

# # Product type imbalance recommendations
# major_product_type = product_type_counts.index[0]
# major_percentage = (product_type_counts.iloc[0] / total_products) * 100

# if major_percentage > 50:
#     print("\nProduct Type Imbalance Recommendations:")
#     print(f"• Dataset is heavily skewed toward {major_product_type} ({major_percentage:.1f}%)")
#     print("• Consider creating separate embedding models for major categories")
#     print("• Implement category-specific search weightings")
#     print("• Ensure evaluation metrics account for this imbalance")
#     print("• Sample data more evenly across categories if possible")

# # Text field recommendations
# print("\nText Field Processing Recommendations:")

# for field in text_fields:
#     if field in df.columns:
#         length_col = f'{field}_length'
#         p95 = df[length_col].quantile(0.95)
#         p99 = df[length_col].quantile(0.99)
#         max_len = df[length_col].max()
        
#         if max_len > 5000:
#             print(f"\n• {field}:")
#             print(f"  - Extreme outliers detected (max: {max_len} chars)")
#             print(f"  - Consider truncating to {p95:.0f} chars (95th percentile)")
#             print(f"  - Or implement text summarization for extremely long content")
            
#         if p95 > 512:
#             print(f"\n• {field}:")
#             print(f"  - Too long for standard embedding models (95th percentile: {p95:.0f} chars)")
#             print(f"  - Consider chunking into smaller segments")
#             print(f"  - Use models with larger context windows (e.g., 4096 tokens)")
#             print(f"  - Implement hierarchical embeddings for retrieval")

# # Save analysis summary to a text file
# analysis_summary_path = '/kaggle/working/quantitative_analysis_summary.txt'

# with open(analysis_summary_path, 'w') as f:
#     f.write("="*80 + "\n")
#     f.write("QUANTITATIVE ANALYSIS SUMMARY\n")
#     f.write("="*80 + "\n\n")
    
#     f.write("1. PRODUCT TYPE DISTRIBUTION\n")
#     f.write("-" * 40 + "\n")
#     f.write(f"Total unique product types: {len(product_type_counts)}\n\n")
    
#     f.write("Top 10 product types:\n")
#     for idx, (product_type, count) in enumerate(product_type_counts.head(10).items()):
#         percentage = (count / total_products) * 100
#         f.write(f"{idx+1}. {product_type}: {count} products ({percentage:.2f}%)\n")
    
#     f.write(f"\nTop 3 product types account for {top_3_percentage:.2f}% of all products\n")
#     f.write(f"Gini coefficient: {gini:.4f} (0 = perfect equality, 1 = perfect inequality)\n\n")
    
#     f.write("2. TEXT FIELD CHARACTERISTICS\n")
#     f.write("-" * 40 + "\n")
    
#     for field in text_fields:
#         if field in df.columns:
#             length_col = f'{field}_length'
#             word_count_col = f'{field}_word_count'
            
#             non_empty = (df[length_col] > 0).sum()
#             percentage = (non_empty / len(df)) * 100
            
#             mean_len = df[length_col].mean()
#             median_len = df[length_col].median()
#             max_len = df[length_col].max()
#             p95 = df[length_col].quantile(0.95)
            
#             mean_words = df[word_count_col].mean()
#             median_words = df[word_count_col].median()
#             p95_words = df[word_count_col].quantile(0.95)
            
#             f.write(f"\n{field}:\n")
#             f.write(f"  Availability: {non_empty}/{len(df)} ({percentage:.2f}%)\n")
#             f.write(f"  Mean length: {mean_len:.2f} chars, {mean_words:.2f} words\n")
#             f.write(f"  Median length: {median_len:.2f} chars, {median_words:.2f} words\n")
#             f.write(f"  Maximum length: {max_len} chars\n")
#             f.write(f"  95th percentile: {p95:.2f} chars, {p95_words:.2f} words\n")
            
#             token_limit = int(p95_words * 1.3)
#             f.write(f"  Recommended token limit: {token_limit} tokens\n")
    
#     f.write("\n3. RECOMMENDATIONS\n")
#     f.write("-" * 40 + "\n")
    
#     # Add product type recommendations
#     if major_percentage > 50:
#         f.write(f"\nProduct Type Imbalance:\n")
#         f.write(f"• Dataset is heavily skewed toward {major_product_type} ({major_percentage:.1f}%)\n")
#         f.write("• Consider creating separate embedding models for major categories\n")
#         f.write("• Implement category-specific search weightings\n")
    
#     # Add text field recommendations
#     f.write("\nText Field Processing:\n")
#     for field in text_fields:
#         if field in df.columns:
#             length_col = f'{field}_length'
#             p95 = df[length_col].quantile(0.95)
#             max_len = df[length_col].max()
            
#             if max_len > 5000 or p95 > 512:
#                 f.write(f"\n• {field}:\n")
                
#                 if max_len > 5000:
#                     f.write(f"  - Extreme outliers detected (max: {max_len} chars)\n")
#                     f.write(f"  - Consider truncating to {p95:.0f} chars (95th percentile)\n")
                
#                 if p95 > 512:
#                     f.write(f"  - Too long for standard embedding models\n")
#                     f.write(f"  - Consider chunking or summarization\n")

# print(f"\nDetailed analysis summary saved to: {analysis_summary_path}")

In [ ]:
# # Create output file names with appropriate prefixes
# sample_prefix = "sample_" if SAMPLE_MODE else ""
# output_dir = '/kaggle/working'

# # Save the data
# df.to_csv(f'{output_dir}/{sample_prefix}abo_products_english_only.csv', index=False)
# print(f"\nSaved {'sample' if SAMPLE_MODE else 'full'} dataset to {output_dir}/{sample_prefix}abo_products_english_only.csv")

In [ ]:
# # Save a version with just essential fields for embedding
# essential_cols = [
#     'item_id', 'domain_name', 'marketplace', 'country',
#     'item_name', 'brand', 'bullet_point', 'product_description',
#     'item_keywords', 'product_type', 'color', 'material', 'style',
#     'main_image_id', 'main_image_url'
# ]

# essential_cols = [col for col in essential_cols if col in df.columns]
# df_essential = df[essential_cols].copy()
# df_essential.to_csv(f'{output_dir}/{sample_prefix}abo_products_embedding_essentials.csv', index=False)
# print(f"Saved essential fields dataset to {output_dir}/{sample_prefix}abo_products_embedding_essentials.csv")

In [ ]:
# # Create a dataset of products with images
# products_with_images = df[~df['main_image_url'].isna()].copy()
# products_with_images[essential_cols].to_csv(f'{output_dir}/abo_products_with_images.csv', index=False)
# print(f"Saved products with images ({len(products_with_images)}) to {output_dir}/abo_products_with_images.csv")

In [ ]:
# Print data types of each column
print("\nData types summary:")
print(df.dtypes.value_counts())